# 01 数据清洗

**目标**：把一份「能打开但跑不了模型」的脏数据，整理成可用于训练的数据。

**数据集**：模拟的电信客户流失数据，1500 条左右，包含四类常见问题：
1. 数值列里混入空字符串和 `N/A`
2. 年龄出现 999、月费出现负数
3. 同一客户重复出现
4. 特征缺失

In [7]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
df = pd.read_csv('../../data/raw/customer_churn.csv', encoding='utf-8-sig')
print(df.shape)
df.head()

(1545, 10)


,customer_id,age,tenure_months,monthly_charge,total_charge,contract_type,payment_method,support_calls,is_active,churn
0,C01225,22.0,43,35.39,1521.77,一年合约,电子支票,1,0,0
1,C01388,53.0,43,140.14,6026.02,按月付费,信用卡,10,0,1
2,C00940,31.0,62,104.32,6467.84,两年合约,银行转账,10,1,0
3,C00624,56.0,71,40.84,2899.64,按月付费,电子支票,8,1,0
4,C01265,48.0,19,103.23,1961.37,两年合约,银行转账,5,1,0


## 1. 先看清问题：类型与缺失

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1545 entries, 0 to 1544
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     1545 non-null   str    
 1   age             1470 non-null   float64
 2   tenure_months   1545 non-null   int64  
 3   monthly_charge  1475 non-null   float64
 4   total_charge    1475 non-null   float64
 5   contract_type   1545 non-null   str    
 6   payment_method  1545 non-null   str    
 7   support_calls   1545 non-null   int64  
 8   is_active       1545 non-null   int64  
 9   churn           1545 non-null   int64  
dtypes: float64(3), int64(4), str(3)
memory usage: 164.7 KB


In [5]:
df.isna().sum().sort_values(ascending=False).head(10)

age               75
monthly_charge    70
total_charge      70
customer_id        0
tenure_months      0
contract_type      0
payment_method     0
support_calls      0
is_active          0
churn              0
dtype: int64

## 2. 类型问题：`N/A` 让数值列变成了 object

`errors='coerce'` 会把无法转换的值变成 NaN，而不是直接报错。

In [4]:
for col in ['age', 'monthly_charge', 'total_charge']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[['age', 'monthly_charge', 'total_charge']].dtypes

age               float64
monthly_charge    float64
total_charge      float64
dtype: object

## 3. 异常值：业务规则比统计规则更可靠

In [6]:
print('年龄 > 100:', (df['age'] > 100).sum())
print('月费 < 0  :', (df['monthly_charge'] < 0).sum())

df.loc[df['age'] > 100, 'age'] = np.nan
df.loc[df['monthly_charge'] < 0, 'monthly_charge'] = np.nan
df.loc[df['total_charge'] < 0, 'total_charge'] = np.nan
print('已把异常值置为缺失，交给后续填充策略处理')

年龄 > 100: 29
月费 < 0  : 31
已把异常值置为缺失，交给后续填充策略处理


## 4. 重复行：按业务主键去重

In [6]:
before = len(df)
df = df.drop_duplicates(subset=['customer_id'])
print(f'删除重复 {before - len(df)} 行，剩余 {len(df)} 行')

删除重复 45 行，剩余 1500 行


## 5. 清洗前后对比

In [7]:
print('缺失值统计:')
print(df[['age', 'monthly_charge', 'total_charge']].isna().sum())
print()
print('流失比例:', f"{df['churn'].mean():.2%}")

缺失值统计:
age               102
monthly_charge    100
total_charge       67
dtype: int64

流失比例: 17.20%


## 小结

- 清洗的核心不是「用哪个函数」，而是**先定义业务上什么叫不合理**
- 异常值处理有两种策略：置为缺失后填充，或直接剔除。本例选前者，避免丢失样本
- 下一步：把清洗逻辑固化成脚本 `src/data_pipeline.py`，让流程可复现